In [2]:
import pandas as pd
import numpy as np
calendar = pd.read_csv("calendar_processed.csv",parse_dates=['date'])
sales_train = pd.read_csv('sales_train.csv',parse_dates=['date'])
sales_test = pd.read_csv('sales_test.csv',parse_dates=['date'])
inventory = pd.read_csv('inventory.csv')

## 合并数据

In [3]:
#合并 sales train, sales test与inventory
inventory_subset = inventory[['unique_id', 'product_unique_id', 'warehouse']]

# Merge with sales_train and sales_test
sales_train = pd.merge(sales_train, inventory_subset, how='left', on=['unique_id', 'warehouse'])
sales_test = pd.merge(sales_test, inventory_subset, how='left', on=['unique_id', 'warehouse'])


In [4]:
#处理sales na step1:什么都有只没有sales
sales_train.loc[sales_train['total_orders'].notna() & sales_train['sell_price_main'].notna(), 'sales'] = sales_train['sales'].fillna(0)

## 小智helpme

In [3]:
#补全sales_test里的缺少日期来merge它和calender,如果warehouse那天补上test里unique_id对应的row
# Step 1: Get all unique combinations of warehouse and date from the calendar
unique_combinations_calendar = calendar[['warehouse', 'date']].drop_duplicates()

# Step 2: Get all combinations of warehouse and date already in sales_train
unique_combinations_sales_train = sales_train[['warehouse', 'date']].drop_duplicates()

# Step 3: Find missing rows (combinations of warehouse and date)
missing_combinations = unique_combinations_calendar.merge(
    unique_combinations_sales_train,
    on=['warehouse', 'date'],
    how='left',
    indicator=True
).query('_merge == "left_only"').drop('_merge', axis=1)

# Step 4: For each missing combination, create a row for every unique_id in sales_test
new_rows = missing_combinations.merge(
    sales_test[['unique_id']], 
    how='cross'
)

# Step 5: Append the new rows to sales_train
sales_train = pd.concat([sales_train, new_rows], ignore_index=True)

# Merge calendar with sales_train on 'warehouse' and 'date'
merged_data = pd.merge(sales_train, calendar, on=['warehouse', 'date'], how='left')

: 

In [ ]:
#sales_test和sales_trains处理,drop availability,合并discounts,和calendar merge
def process_sales_data(sales_data, calendar):
    discount_cols = [f"type_{i}_discount" for i in range(7)]
    sales_data = pd.merge(sales_data, calendar, how='left', on=['date', 'warehouse'])
    sales_data["min_discount"] = sales_data[discount_cols].min(axis=1).fillna(1)
    
    # Drop discount columns
    drop_cols = discount_cols + ["availability"] if "availability" in sales_data.columns else discount_cols
    return sales_data.drop(columns=drop_cols, errors='ignore')

sales_train = process_sales_data(sales_train, calendar)
sales_test = process_sales_data(sales_test, calendar)

sales_test.to_csv('processed_sales_test.csv', index=False)

In [ ]:
# One-hot encode the 'warehouse' column
encoded_warehouses = pd.get_dummies(sales_train['warehouse'], prefix='warehouse')
sales_train_encoded = pd.concat([sales_train.drop('warehouse', axis=1), encoded_warehouses], axis=1)

## 继续处理sales na

In [ ]:
#处理sales na step2:shop_closed == 1
# Fill sales with 0 if shop_closed == 1 and sales is NaN
sales_train.loc[(sales_train['shop_closed'] == 1) & (sales_train['sales'].isna()), 'sales'] = 0

# Fill missing price using the previous day's price within the same warehouse and unique_id
sales_train['price'] = sales_train.groupby(['warehouse', 'unique_id'])['price'].ffill()

In [ ]:
#处理sales na step3:疫情
# Get all relevant dates from missing_combinations
missing_dates = missing_combinations['date'].unique()

# Define the conditions for sales_train where sales should be filled
condition_munich = (sales_train['warehouse'] == 'Munich_1') & (sales_train['date'].isin(missing_dates))
condition_frankfurt = (sales_train['warehouse'] == 'Frankfurt_1') & (sales_train['date'].isin(missing_dates))

# Combine conditions
condition = condition_munich | condition_frankfurt

# Compute the average sales for the same product_unique_id and date across other warehouses
avg_sales = sales_train.groupby(['product_unique_id', 'date'])['sales'].transform(lambda x: x.mean())

# Fill sales where the condition is met and sales is NaN
sales_train.loc[condition & sales_train['sales'].isna(), 'sales'] = avg_sales

In [ ]:
#处理sales na step3:其他
# Step 2: For the rest of missing_dates in missing_combinations:
# - Set sales to 0
# - Fill total_orders using same day's value within the same warehouse, then use previous day's value if still missing
# - Fill price using previous day's price within the same warehouse and unique_id

# Identify remaining missing sales
remaining_condition = sales_train['date'].isin(missing_dates) & sales_train['sales'].isna()

# Fill sales with 0
sales_train.loc[remaining_condition, 'sales'] = 0

# Fill total_orders using same day's value within the same warehouse
sales_train['total_orders'] = sales_train.groupby(['date', 'warehouse'])['total_orders'].transform(lambda x: x.fillna(method='ffill'))

# Fill remaining missing total_orders using previous day's value within the same warehouse
sales_train['total_orders'] = sales_train.groupby('warehouse')['total_orders'].ffill()

# Fill price using previous day's price within the same warehouse and unique_id
sales_train['price'] = sales_train.groupby(['warehouse', 'unique_id'])['price'].ffill()

In [ ]:
print(sales_train[sales_train['sales'].isna()])